In [1]:
import os, json, shutil
from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from tqdm import tqdm
from glob import glob
import pandas as pd
from scipy.ndimage import label as cc_label


SPLITS_2020 = Path(r"D:\master_experiments\data\splits\BraTS2020_Splits")
SPLITS_2024 = Path(r"D:\master_experiments\data\splits\BraTS2024_Splits")

NNUNET_2020_BASE = Path(r"D:\master_experiments\models_configs\nnUNet_2020")
NNUNET_2024_BASE = Path(r"D:\master_experiments\models_configs\nnUNet_2024")

DATASET_2020_ID, DATASET_2020_NAME = 902, "BraTS2020_Baseline_4ch"
DATASET_2024_ID, DATASET_2024_NAME = 903, "BraTS2024_Baseline_4ch"

IMAGES_TS_2020 = NNUNET_2020_BASE / "nnUNet_raw" / f"Dataset{DATASET_2020_ID:03d}_{DATASET_2020_NAME}" / "imagesTs"
IMAGES_TS_2024 = NNUNET_2024_BASE / "nnUNet_raw" / f"Dataset{DATASET_2024_ID:03d}_{DATASET_2024_NAME}" / "imagesTs"

CROSS_BASE        = Path(r"D:\master_experiments\models_configs\cross_eval\nnUNet")
PRED_2020_ON_2024 = CROSS_BASE / "preds_2020_on_2024"
PRED_2024_ON_2020 = CROSS_BASE / "preds_2024_on_2020"
LOGS              = CROSS_BASE / "logs"
for p in [PRED_2020_ON_2024, PRED_2024_ON_2020, LOGS]:
    p.mkdir(parents=True, exist_ok=True)

print("Paths OK")
print(f"  imagesTs 2020 existe: {IMAGES_TS_2020.exists()}")
print(f"  imagesTs 2024 existe: {IMAGES_TS_2024.exists()}")
print(f"  Splits 2020 existe:  {SPLITS_2020.exists()}")
print(f"  Splits 2024 existe:  {SPLITS_2024.exists()}")

Paths OK
  imagesTs 2020 existe: True
  imagesTs 2024 existe: True
  Splits 2020 existe:  True
  Splits 2024 existe:  True


In [2]:
with open(SPLITS_2020 / "splits_metadata.json", "r", encoding="utf-8") as f:
    meta_2020 = json.load(f)
with open(SPLITS_2024 / "splits_metadata.json", "r", encoding="utf-8") as f:
    meta_2024 = json.load(f)

test_ids_2020 = meta_2020["ids"]["test"]
test_ids_2024 = meta_2024["ids"]["test"]

print(f"Test BraTS2020: {len(test_ids_2020)} casos")
print(f"Test BraTS2024: {len(test_ids_2024)} casos")

Test BraTS2020: 53 casos
Test BraTS2024: 203 casos


In [3]:
def case_dir_2020(cid): return SPLITS_2020 / "test" / cid
def case_dir_2024(cid): return SPLITS_2024 / "test" / cid

def find_seg(d):
    for cand in [d / "seg.nii.gz", d / "seg.nii"]:
        if cand.exists(): return cand
    cands = sorted(d.glob("*seg*.nii*"))
    if not cands:
        raise FileNotFoundError(f"seg nao encontrada em {d}")
    return cands[0]

def load_arr(p):
    return np.asanyarray(nib.load(str(p)).dataobj)

def dice(a, b):
    inter = np.count_nonzero(a & b)
    denom = np.count_nonzero(a) + np.count_nonzero(b)
    return 1.0 if denom == 0 else (2.0 * inter / denom)

def get_pred_path(pred_dir, cid):
    p = pred_dir / f"{cid}.nii.gz"
    if p.exists(): return p
    p = pred_dir / f"{cid}.nii"
    if p.exists(): return p
    raise FileNotFoundError(f"Pred nao encontrada: {pred_dir}/{cid}")

print("Helpers OK")

Helpers OK


In [4]:
# Direcao A: nnUNet treinado em BraTS2020 -> predito sobre teste BraTS2024 

os.environ["nnUNet_raw"]          = str(NNUNET_2020_BASE / "nnUNet_raw")
os.environ["nnUNet_preprocessed"] = str(NNUNET_2020_BASE / "nnUNet_preprocessed")
os.environ["nnUNet_results"]      = str(NNUNET_2020_BASE / "nnUNet_results")

!nnUNetv2_predict -i "{IMAGES_TS_2024}" -o "{PRED_2020_ON_2024}" -d {DATASET_2020_ID} -c 3d_fullres -f 0 -tr nnUNetTrainer_250epochs


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 203 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 203 cases that I would like to predict

Predicting BraTS-GLI-00005-100:
perform_everything_on_device: True
sending off prediction to background worker for resampling and export
done with BraTS-GLI-00005-100

Predicting BraTS-GLI-00008-101:
perform_everything_on_device: True
sending off prediction to background worker for resampling and export
done with BraTS-GLI-00008-101

Predicting BraTS-GLI-00020-100:
perform_everything_on_device: True
sending off prediction to b


100%|##########| 8/8 [00:03<00:00,  2.12it/s]

100%|##########| 4/4 [00:01<00:00,  3.44it/s]

100%|##########| 8/8 [00:02<00:00,  3.20it/s]

100%|##########| 8/8 [00:02<00:00,  3.18it/s]

100%|##########| 8/8 [00:02<00:00,  3.16it/s]

100%|##########| 8/8 [00:02<00:00,  3.17it/s]

100%|##########| 8/8 [00:02<00:00,  3.08it/s]

100%|##########| 8/8 [00:02<00:00,  3.21it/s]

100%|##########| 8/8 [00:02<00:00,  3.21it/s]

100%|##########| 8/8 [00:02<00:00,  3.23it/s]

100%|##########| 8/8 [00:02<00:00,  3.21it/s]

100%|##########| 8/8 [00:02<00:00,  3.20it/s]

100%|##########| 8/8 [00:02<00:00,  3.21it/s]

100%|##########| 8/8 [00:02<00:00,  3.21it/s]

100%|##########| 4/4 [00:01<00:00,  3.40it/s]

100%|##########| 8/8 [00:02<00:00,  3.21it/s]

100%|##########| 8/8 [00:02<00:00,  3.20it/s]

100%|##########| 8/8 [00:02<00:00,  3.21it/s]

100%|##########| 4/4 [00:01<00:00,  3.44it/s]

100%|##########| 8/8 [00:02<00:00,  3.11it/s]

100%|##########| 8/8 [00:02<00:00,  2.99it/s]

100%|#######

In [5]:
# Metrica 1 (A): Dice GLOBAL - modelo 2020 sobre teste 2024

rows = []
for cid in tqdm(test_ids_2024, desc="Dice global por caso (A: 2020->2024)"):
    gt = load_arr(find_seg(case_dir_2024(cid))).astype(np.int16)  
    pr = load_arr(get_pred_path(PRED_2020_ON_2024, cid)).astype(np.int16) 

    wt = dice(((gt==1)|(gt==2)|(gt==3)), ((pr==1)|(pr==2)|(pr==3)))
    tc = dice(((gt==1)|(gt==3)),         ((pr==1)|(pr==3)))
    et = dice((gt==3),                   (pr==3))

    rows.append({"id": cid, "dice_WT": wt, "dice_TC": tc, "dice_ET": et})

df_A = pd.DataFrame(rows)
df_A.to_csv(LOGS / "cross_A_2020_on_2024_dice.csv", index=False)
df_A[["dice_WT","dice_TC","dice_ET"]].describe().round(4)

Dice global por caso (A: 2020->2024): 100%|██████████| 203/203 [00:26<00:00,  7.55it/s]


,dice_WT,dice_TC,dice_ET
count,203.0000,203.0000,203.0000
mean,0.7414,0.4426,0.6045
std,0.2416,0.4004,0.3766
min,0.0000,0.0000,0.0000
25%,0.6851,0.0000,0.2809
50%,0.8226,0.4674,0.7270
75%,0.8985,0.8552,0.9313
max,0.9829,1.0000,1.0000


In [6]:
# Analise condicional 1/3 (A): Frequencia de presenca das regioes compostas no GT 2024

presence_rows = []
for cid in tqdm(test_ids_2024, desc="Presenca GT por caso (A)"):
    gt = load_arr(find_seg(case_dir_2024(cid))).astype(np.int16)
    presence_rows.append({
        "id":     cid,
        "has_WT": bool(((gt==1)|(gt==2)|(gt==3)).any()),
        "has_TC": bool(((gt==1)|(gt==3)).any()),
        "has_ET": bool((gt==3).any()),
    })
pres_df_A = pd.DataFrame(presence_rows)
df_full_A = df_A.merge(pres_df_A, on="id")

df_presence_A = pd.DataFrame([
    {"region":     col,
     "n_presente": int(df_full_A[col].sum()),
     "n_total":    len(df_full_A),
     "pct":        100.0 * df_full_A[col].sum() / len(df_full_A)}
    for col in ["has_WT", "has_TC", "has_ET"]
])
df_presence_A.round(2)

Presenca GT por caso (A): 100%|██████████| 203/203 [00:17<00:00, 11.84it/s]


,region,n_presente,n_total,pct
0,has_WT,203,203,100.00
1,has_TC,152,203,74.88
2,has_ET,150,203,73.89


In [7]:
# Analise condicional 2/3 (A): Dice CONDICIONAL - apenas casos com regiao presente no GT

pairs = [("dice_WT","has_WT"), ("dice_TC","has_TC"), ("dice_ET","has_ET")]
df_conditional_A = pd.DataFrame({
    col: df_full_A.loc[df_full_A[has], col].reset_index(drop=True)
    for col, has in pairs
})
df_conditional_A.to_csv(LOGS / "cross_A_2020_on_2024_conditional.csv", index=False)
df_conditional_A.describe().round(4)

,dice_WT,dice_TC,dice_ET
count,203.0000,152.0000,150.0000
mean,0.7414,0.4990,0.5781
std,0.2416,0.3667,0.3349
min,0.0000,0.0000,0.0000
25%,0.6851,0.0671,0.3341
50%,0.8226,0.5986,0.6945
75%,0.8985,0.8505,0.8629
max,0.9829,0.9855,0.9761


In [8]:
# Analise condicional 3/3 (A): Bimodalidade

df_bimodality_A = pd.DataFrame([
    {"region":         col,
     "pct_dice_zero": float((df_full_A[col] == 0.0).mean()) * 100,
     "pct_dice_one":  float((df_full_A[col] >= 0.999).mean()) * 100}
    for col, _ in pairs
])
df_bimodality_A.to_csv(LOGS / "cross_A_2020_on_2024_bimodality.csv", index=False)
df_bimodality_A.round(2)

,region,pct_dice_zero,pct_dice_one
0,dice_WT,4.43,0.00
1,dice_TC,28.08,6.90
2,dice_ET,18.23,17.73


In [9]:
# Metrica 3 (A): Lesion-wise Dice

LW_MIN_SIZE = 50

def lesion_wise_dice(gt_mask: np.ndarray, pr_mask: np.ndarray, min_size: int = LW_MIN_SIZE) -> float:
    structure = np.ones((3, 3, 3), dtype=int)
    if not gt_mask.any() and not pr_mask.any():
        return 1.0
    gt_lab, n_gt = cc_label(gt_mask.astype(bool), structure=structure)
    pr_lab, n_pr = cc_label(pr_mask.astype(bool), structure=structure)
    dice_list = []
    matched_pred = set()
    for g in range(1, n_gt + 1):
        gt_l = (gt_lab == g)
        if gt_l.sum() < min_size:
            continue
        overlap_ids = np.unique(pr_lab[gt_l])
        overlap_ids = overlap_ids[overlap_ids > 0]
        if len(overlap_ids) == 0:
            dice_list.append(0.0)
        else:
            pr_match = np.isin(pr_lab, overlap_ids)
            inter = np.logical_and(gt_l, pr_match).sum()
            denom = gt_l.sum() + pr_match.sum()
            dice_list.append(2.0 * inter / denom if denom > 0 else 0.0)
            matched_pred.update(int(p) for p in overlap_ids)
    for p in range(1, n_pr + 1):
        if p in matched_pred:
            continue
        pr_l = (pr_lab == p)
        if pr_l.sum() < min_size:
            continue
        dice_list.append(0.0)
    return float(np.mean(dice_list)) if dice_list else 1.0


lw_rows = []
for cid in tqdm(test_ids_2024, desc="Lesion-wise Dice por caso (A)"):
    gt = load_arr(find_seg(case_dir_2024(cid))).astype(np.int16)
    pr = load_arr(get_pred_path(PRED_2020_ON_2024, cid)).astype(np.int16)

    gt_wt = (gt==1)|(gt==2)|(gt==3)
    pr_wt = (pr==1)|(pr==2)|(pr==3)
    gt_tc = (gt==1)|(gt==3)
    pr_tc = (pr==1)|(pr==3)
    gt_et = (gt==3)
    pr_et = (pr==3)

    lw_rows.append({
        "id":     cid,
        "lwd_WT": lesion_wise_dice(gt_wt, pr_wt),
        "lwd_TC": lesion_wise_dice(gt_tc, pr_tc),
        "lwd_ET": lesion_wise_dice(gt_et, pr_et),
    })

lw_df_A = pd.DataFrame(lw_rows)
lw_df_A.to_csv(LOGS / "cross_A_2020_on_2024_lwd.csv", index=False)
lw_df_A[["lwd_WT","lwd_TC","lwd_ET"]].describe().round(4)

Lesion-wise Dice por caso (A): 100%|██████████| 203/203 [02:28<00:00,  1.37it/s]


,lwd_WT,lwd_TC,lwd_ET
count,203.0000,203.0000,203.0000
mean,0.5258,0.4111,0.6202
std,0.3036,0.3890,0.3662
min,0.0000,0.0000,0.0000
25%,0.2771,0.0000,0.3372
50%,0.4401,0.3196,0.7302
75%,0.8531,0.8226,1.0000
max,0.9829,1.0000,1.0000


In [10]:
# Direcao B: nnUNet treinado em BraTS2024 -> predito sobre teste BraTS2020 
# Configura env vars apontando para o modelo 2024.

os.environ["nnUNet_raw"]          = str(NNUNET_2024_BASE / "nnUNet_raw")
os.environ["nnUNet_preprocessed"] = str(NNUNET_2024_BASE / "nnUNet_preprocessed")
os.environ["nnUNet_results"]      = str(NNUNET_2024_BASE / "nnUNet_results")

!nnUNetv2_predict -i "{IMAGES_TS_2020}" -o "{PRED_2024_ON_2020}" -d {DATASET_2024_ID} -c 3d_fullres -f 0 -tr nnUNetTrainer_250epochs


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 53 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 53 cases that I would like to predict

Predicting BraTS20_Training_006:
perform_everything_on_device: True
sending off prediction to background worker for resampling and export
done with BraTS20_Training_006

Predicting BraTS20_Training_008:
perform_everything_on_device: True
sending off prediction to background worker for resampling and export
done with BraTS20_Training_008

Predicting BraTS20_Training_014:
perform_everything_on_device: True
sending off prediction t


100%|##########| 8/8 [00:03<00:00,  2.01it/s]

100%|##########| 8/8 [00:02<00:00,  2.77it/s]

100%|##########| 8/8 [00:02<00:00,  2.77it/s]

100%|##########| 8/8 [00:02<00:00,  2.81it/s]

100%|##########| 8/8 [00:02<00:00,  2.78it/s]

100%|##########| 8/8 [00:02<00:00,  2.77it/s]

100%|##########| 8/8 [00:03<00:00,  2.57it/s]

100%|##########| 8/8 [00:03<00:00,  2.47it/s]

100%|##########| 8/8 [00:02<00:00,  2.72it/s]

100%|##########| 8/8 [00:02<00:00,  2.79it/s]

100%|##########| 4/4 [00:01<00:00,  2.98it/s]

100%|##########| 8/8 [00:02<00:00,  2.79it/s]

100%|##########| 8/8 [00:02<00:00,  2.76it/s]

100%|##########| 8/8 [00:02<00:00,  2.81it/s]

100%|##########| 8/8 [00:02<00:00,  2.79it/s]

100%|##########| 8/8 [00:02<00:00,  2.76it/s]

100%|##########| 8/8 [00:02<00:00,  2.81it/s]

100%|##########| 8/8 [00:02<00:00,  2.79it/s]

100%|##########| 4/4 [00:01<00:00,  2.95it/s]

100%|##########| 4/4 [00:01<00:00,  2.95it/s]

100%|##########| 8/8 [00:02<00:00,  2.80it/s]

100%|#######

In [11]:
# Metrica 1 (B): Dice GLOBAL - modelo 2024 sobre teste 2020

rows = []
for cid in tqdm(test_ids_2020, desc="Dice global por caso (B: 2024->2020)"):
    gt = load_arr(find_seg(case_dir_2020(cid))).astype(np.int16)
    gt[gt == 4] = 3  
    pr = load_arr(get_pred_path(PRED_2024_ON_2020, cid)).astype(np.int16)
    pr[pr == 4] = 0  

    wt = dice(((gt==1)|(gt==2)|(gt==3)), ((pr==1)|(pr==2)|(pr==3)))
    tc = dice(((gt==1)|(gt==3)),         ((pr==1)|(pr==3)))
    et = dice((gt==3),                   (pr==3))

    rows.append({"id": cid, "dice_WT": wt, "dice_TC": tc, "dice_ET": et})

df_B = pd.DataFrame(rows)
df_B.to_csv(LOGS / "cross_B_2024_on_2020_dice.csv", index=False)
df_B[["dice_WT","dice_TC","dice_ET"]].describe().round(4)

Dice global por caso (B: 2024->2020): 100%|██████████| 53/53 [00:06<00:00,  8.71it/s]


,dice_WT,dice_TC,dice_ET
count,53.0000,53.0000,53.0000
mean,0.8811,0.7825,0.8127
std,0.0780,0.2997,0.1889
min,0.6594,0.0000,0.0000
25%,0.8319,0.8282,0.7937
50%,0.9062,0.9170,0.8815
75%,0.9381,0.9412,0.9126
max,0.9689,0.9782,0.9574


In [12]:
# Analise condicional 1/3 (B): Frequencia de presenca das regioes compostas no GT 2020

presence_rows = []
for cid in tqdm(test_ids_2020, desc="Presenca GT por caso (B)"):
    gt = load_arr(find_seg(case_dir_2020(cid))).astype(np.int16)
    gt[gt == 4] = 3
    presence_rows.append({
        "id":     cid,
        "has_WT": bool(((gt==1)|(gt==2)|(gt==3)).any()),
        "has_TC": bool(((gt==1)|(gt==3)).any()),
        "has_ET": bool((gt==3).any()),
    })
pres_df_B = pd.DataFrame(presence_rows)
df_full_B = df_B.merge(pres_df_B, on="id")

df_presence_B = pd.DataFrame([
    {"region":     col,
     "n_presente": int(df_full_B[col].sum()),
     "n_total":    len(df_full_B),
     "pct":        100.0 * df_full_B[col].sum() / len(df_full_B)}
    for col in ["has_WT", "has_TC", "has_ET"]
])
df_presence_B.round(2)

Presenca GT por caso (B): 100%|██████████| 53/53 [00:02<00:00, 24.30it/s]


,region,n_presente,n_total,pct
0,has_WT,53,53,100.0
1,has_TC,53,53,100.0
2,has_ET,53,53,100.0


In [13]:
# Analise condicional 2/3 (B): Dice CONDICIONAL

pairs = [("dice_WT","has_WT"), ("dice_TC","has_TC"), ("dice_ET","has_ET")]
df_conditional_B = pd.DataFrame({
    col: df_full_B.loc[df_full_B[has], col].reset_index(drop=True)
    for col, has in pairs
})
df_conditional_B.to_csv(LOGS / "cross_B_2024_on_2020_conditional.csv", index=False)
df_conditional_B.describe().round(4)

,dice_WT,dice_TC,dice_ET
count,53.0000,53.0000,53.0000
mean,0.8811,0.7825,0.8127
std,0.0780,0.2997,0.1889
min,0.6594,0.0000,0.0000
25%,0.8319,0.8282,0.7937
50%,0.9062,0.9170,0.8815
75%,0.9381,0.9412,0.9126
max,0.9689,0.9782,0.9574


In [14]:
# Analise condicional 3/3 (B): Bimodalidade

df_bimodality_B = pd.DataFrame([
    {"region":         col,
     "pct_dice_zero": float((df_full_B[col] == 0.0).mean()) * 100,
     "pct_dice_one":  float((df_full_B[col] >= 0.999).mean()) * 100}
    for col, _ in pairs
])
df_bimodality_B.to_csv(LOGS / "cross_B_2024_on_2020_bimodality.csv", index=False)
df_bimodality_B.round(2)

,region,pct_dice_zero,pct_dice_one
0,dice_WT,0.00,0.0
1,dice_TC,1.89,0.0
2,dice_ET,3.77,0.0


In [15]:
# Metrica 3 (B): Lesion-wise Dice

lw_rows = []
for cid in tqdm(test_ids_2020, desc="Lesion-wise Dice por caso (B)"):
    gt = load_arr(find_seg(case_dir_2020(cid))).astype(np.int16)
    gt[gt == 4] = 3
    pr = load_arr(get_pred_path(PRED_2024_ON_2020, cid)).astype(np.int16)
    pr[pr == 4] = 0 

    gt_wt = (gt==1)|(gt==2)|(gt==3)
    pr_wt = (pr==1)|(pr==2)|(pr==3)
    gt_tc = (gt==1)|(gt==3)
    pr_tc = (pr==1)|(pr==3)
    gt_et = (gt==3)
    pr_et = (pr==3)

    lw_rows.append({
        "id":     cid,
        "lwd_WT": lesion_wise_dice(gt_wt, pr_wt),
        "lwd_TC": lesion_wise_dice(gt_tc, pr_tc),
        "lwd_ET": lesion_wise_dice(gt_et, pr_et),
    })

lw_df_B = pd.DataFrame(lw_rows)
lw_df_B.to_csv(LOGS / "cross_B_2024_on_2020_lwd.csv", index=False)
lw_df_B[["lwd_WT","lwd_TC","lwd_ET"]].describe().round(4)

Lesion-wise Dice por caso (B): 100%|██████████| 53/53 [00:49<00:00,  1.07it/s]


,lwd_WT,lwd_TC,lwd_ET
count,53.0000,53.0000,53.0000
mean,0.6623,0.7462,0.7512
std,0.2819,0.3079,0.2420
min,0.0758,0.0000,0.0000
25%,0.4415,0.6562,0.6931
50%,0.7938,0.9076,0.8628
75%,0.9175,0.9387,0.9115
max,0.9689,0.9782,0.9574


In [16]:
# Tabela consolidada cross-dataset (medias) - apenas WT, TC, ET

summary = pd.DataFrame([
    {"direcao": "2020 -> 2024",
     "Dice WT": df_A["dice_WT"].mean(),  "Dice TC": df_A["dice_TC"].mean(),  "Dice ET": df_A["dice_ET"].mean(),
     "LWD WT":  lw_df_A["lwd_WT"].mean(), "LWD TC":  lw_df_A["lwd_TC"].mean(), "LWD ET":  lw_df_A["lwd_ET"].mean()},
    {"direcao": "2024 -> 2020",
     "Dice WT": df_B["dice_WT"].mean(),  "Dice TC": df_B["dice_TC"].mean(),  "Dice ET": df_B["dice_ET"].mean(),
     "LWD WT":  lw_df_B["lwd_WT"].mean(), "LWD TC":  lw_df_B["lwd_TC"].mean(), "LWD ET":  lw_df_B["lwd_ET"].mean()},
])
summary.to_csv(LOGS / "cross_summary_nnUNet.csv", index=False)
summary.round(4)

,direcao,Dice WT,Dice TC,Dice ET,LWD WT,LWD TC,LWD ET
0,2020 -> 2024,0.7414,0.4426,0.6045,0.5258,0.4111,0.6202
1,2024 -> 2020,0.8811,0.7825,0.8127,0.6623,0.7462,0.7512
